In [ ]:
#%pip install pyserial

In [5]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
  Using cached scikit_learn-1.6.1-cp39-cp39-macosx_12_0_arm64.whl (11.1 MB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
  Using cached scipy-1.13.1-cp39-cp39-macosx_12_0_arm64.whl (30.3 MB)
  Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pandas
%pip install torch
%pip install pyserial


Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-2.3.3-cp39-cp39-macosx_11_0_arm64.whl (10.8 MB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)
     |████████████████████████████████| 510 kB 5.2 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
  Using cached torch-2.8.0-cp39-none-macosx_11_0_arm64.whl (73.6 MB)
  Using cached fsspec-2025.10.0-py3-none-any.whl (200 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
  Using cached networkx-3.2.1-py3-none-any.whl (1.6 MB)
  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
  Attempting uninstall: typing-extensions

In [6]:
import os
import glob
import numpy as np
import pandas as pd
#import matplotlib.pyplot as plt
#import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


import serial
import threading
import time


ImportError: cannot import name 'TypeIs' from 'typing_extensions' (/Users/adithyasuresh/Library/Python/3.9/lib/python/site-packages/typing_extensions.py)

### Loading the Model

In [2]:
# Train REGULARIZED version
class EmotionCNN_Reg(nn.Module):
    def __init__(self, input_channels=14, num_classes=4):
        super().__init__()
        self.conv1 = nn.Conv1d(input_channels, 16, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm1d(16)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm1d(32)
        self.pool2 = nn.MaxPool1d(2)

        self.fc1       = nn.Linear(32 * 31, 64)
        self.dropout1  = nn.Dropout(0.3)
        self.fc2       = nn.Linear(64, 32)
        self.dropout2  = nn.Dropout(0.5)
        self.fc3       = nn.Linear(32, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = F.dropout(x, 0.2)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x



In [3]:
# Load the best CNN model

PATH= 'cnn_without_orientation_data.pth'

model = EmotionCNN_Reg()

model.load_state_dict(torch.load(PATH, weights_only=True))
model.eval()


EmotionCNN_Reg(
  (conv1): Conv1d(14, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=992, out_features=64, bias=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (dropout2): Dropout(p=0.5, inplace=False)
  (fc3): Linear(in_features=32, out_features=4, bias=True)
)

### Live Data Streaming

In [4]:
'''if torch.backends.mps.is_available():
    device = torch.device("mps")  

else:
    device = torch.device("cpu")''' 
device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [1]:
global_mean = [0.000135, 0.000919, -0.001325, 0.000138, -8.5e-05, 0.001069, -0.001142, -0.001113, 0.003192, -0.000292, 0.000753, -0.000517, -0.003865, -0.002841]
global_std  = [0.98748, 0.993947, 0.987439, 0.983629, 0.979366, 0.98413, 0.993598, 0.994275, 0.989484, 0.979971, 0.980799, 0.977189, 0.985282, 0.989895]

In [ ]:
# Start Data streaming
# How is the live data coming in?

COM_PORT='/dev/cu.usbmodem21101'


window_size = 125
overlap_size = int(0.25 * window_size)
step_size = window_size - overlap_size # 94

live_buffer = []               # The shared "bowl"
buffer_lock = threading.Lock() # The "Pause" button to prevent data crashes

def data_collection_thread():
    # Open the serial port inside the producer
    ser = serial.Serial(COM_PORT, 115200)
    ser.reset_input_buffer() # Clear out old junk!
    #correct_columns=21 # for 20 feature model
    correct_columns=15 # for 20 feature model

    

    print("Producer: Listening to Arduino at 30Hz...")
    
    while True:
        try:
            # new code to check if the line has the correct number of columns before next step
            raw_line = ser.readline().decode('utf-8').strip()
            #print(type(raw_line))
            sample_list=[]
            indi_params= raw_line.split(',')

            dropped_indices = [6,7,8,15,16,17]
            filtered_params = [indi_params[i] for i in range(len(indi_params)) if i not in dropped_indices]
            
            #print(len(indi_params))
            if len(filtered_params) == correct_columns:
                sample_list = [float(x) for x in filtered_params] #changed this up to make sure its not a generator and its a list
                #print(sample_list)
                with buffer_lock:
                    live_buffer.append(sample_list)
            else:
                pass
                
        except Exception as e:
            pass


def inference_thread():
    class_labels = {0: "Distracted", 1: "Focus", 2: "Relaxed", 3: "Stress"}    
    print("Consumer: Waiting for 125 rows...")
    model.eval() 
    
    while True:
        data_to_process = None
        
        # Pull from buffer
        with buffer_lock:
            if len(live_buffer) >= window_size:
                data_to_process = list(live_buffer[:window_size])
                print("Data to process: ", data_to_process)
                del live_buffer[:step_size] 
        
        if data_to_process is not None:
            # Convert to numpy (Shape: 125, 21)
            print("IN HERE")
            data_array = np.array(data_to_process,dtype=np.float32)
            
            data_array = data_array[:, 1:] 

            

            mean = np.mean(data_array, axis=0)
            std = np.std(data_array, axis=0)
            data_normalized = (data_array - mean) / (std + 1e-8)

            input_tensor = torch.tensor(data_normalized, dtype=torch.float32)#Convert to Torch Tensor
            
            input_tensor = input_tensor.transpose(0, 1)
            
            input_tensor = input_tensor.unsqueeze(0).to(device)# making it (1, Channels, 125) 

            try:
                with torch.no_grad():
                    output = model(input_tensor)
                    
                    probabilities = torch.softmax(output, dim=1)
                    
                    conf, pred_idx = torch.max(probabilities, dim=1)
                    
                    prediction = pred_idx.item()
                    confidence = conf.item()

                print(f"Result: {class_labels.get(prediction, 'Unknown')} | Conf: {confidence:.2%}")
                
            except Exception as e:
                print(f"Inference Error: {e}")
                print(f"Check if input_tensor shape {input_tensor.shape} matches model!")

        else:
            print("Not enough data yet...")
            time.sleep(0.01)




# Create the worker threads
t1 = threading.Thread(target=data_collection_thread, daemon=True)
t2 = threading.Thread(target=inference_thread, daemon=True)

# Start them
t1.start()
t2.start()

# Keep the main script alive so the threads can run in the background
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Shutting down the system...")




Consumer: Waiting for 125 rows...
Not enough data yet...
Producer: Listening to Arduino at 30Hz...
Not enough data yet...
<class 'str'>
Not enough data yet...
Not enough data yet...
Not enough data yet...
<class 'str'>
Not enough data yet...
Not enough data yet...
Not enough data yet...
<class 'str'>
Not enough data yet...
Not enough data yet...
Not enough data yet...
Not enough data yet...
<class 'str'>
Not enough data yet...
Not enough data yet...
Not enough data yet...
<class 'str'>
Not enough data yet...
Not enough data yet...
Not enough data yet...
<class 'str'>
Not enough data yet...
Not enough data yet...
Not enough data yet...
<class 'str'>
Not enough data yet...
Not enough data yet...
Not enough data yet...
Not enough data yet...
<class 'str'>
Not enough data yet...
Not enough data yet...
Not enough data yet...
<class 'str'>
Not enough data yet...
Not enough data yet...
Not enough data yet...
Not enough data yet...
<class 'str'>
Not enough data yet...
Not enough data yet...
No

### Data Processing

In [ ]:
# Call Data Processing Pipeline

### Model Prediction

In [ ]:
# Live Predictions

